In [0]:
%sql
CREATE OR REPLACE TABLE fact_orders AS
SELECT 
    INT(Order_Item_Id)             AS order_item_id,
    INT(Order_Id)                  AS order_id,
    INT(Order_Customer_Id)         AS customer_id,
    INT(Product_Card_Id)           AS product_id,
    CONCAT_WS('-', Market, Order_Region, Order_Country, Customer_State, Customer_City) AS location_id,
    
    -- Corrected Date Parsing --
    to_date(order_date__DateOrders_, 'M/d/yyyy H:m') AS order_date,
    
    INT(Order_Item_Quantity)       AS order_item_quantity,
    Order_Item_Product_Price       AS order_item_product_price,
    Order_Item_Discount            AS order_item_discount,
    Order_Item_Discount_Rate       AS order_item_discount_rate,
    Order_Item_Total               AS order_item_total,
    Order_Profit_Per_Order         AS order_profit_per_order,
    Sales_per_customer             AS sales_per_customer
FROM table_1;

num_affected_rows,num_inserted_rows


In [0]:
%sql
CREATE OR REPLACE TABLE fact_shipments AS
SELECT 
    -- Creates a clean unique ID for each shipment row
    CONCAT(Order_Id, '-', Order_Item_Id) AS shipment_id,
    INT(Order_Id)                        AS order_id,
    INT(Order_Item_Id)                   AS order_item_id,
    to_date(shipping_date__DateOrders_, 'M/d/yyyy H:m') AS shipping_date,
    
    Delivery_Status                    AS shipment_status,
    Shipping_Mode                        AS shipping_mode,
    INT(Days_for_shipping__real_)          AS days_for_shipping_real,
    INT(Days_for_shipment__scheduled_)     AS days_for_shipment_scheduled,
    INT(Late_delivery_risk)              AS late_delivery_risk

FROM table_1;

num_affected_rows,num_inserted_rows


In [0]:
%sql
select * from fact_shipments limit 3

shipment_id,order_id,order_item_id,shipping_date,shipment_status,shipping_mode,days_for_shipping_real,days_for_shipment_scheduled,late_delivery_risk
77202-180517,77202,180517,2018-02-03,Advance shipping,Standard Class,3,4,0
75939-179254,75939,179254,2018-01-18,Late delivery,Standard Class,5,4,1
75938-179253,75938,179253,2018-01-17,Shipping on time,Standard Class,4,4,0


In [0]:
%sql
CREATE OR REPLACE TABLE dim_products AS
SELECT DISTINCT
    INT(Product_Card_Id)      AS product_id,
    Product_Name              AS product_name,
    Category_Name             AS category_name,
    Department_Name           AS department_name,
    Order_Item_Product_Price  AS product_price
FROM table_1;

num_affected_rows,num_inserted_rows


In [0]:
%sql
CREATE OR REPLACE TABLE dim_locations AS
SELECT DISTINCT
    -- Combine location text to create a unique location key
    CONCAT_WS('-', Market, Order_Region, Order_Country, Customer_State, Customer_City) AS location_id,
    
    Market                    AS market,
    Order_Region              AS order_region,
    Order_Country             AS order_country,
    Customer_State            AS customer_state,
    Customer_City             AS customer_city,
    Latitude                  AS latitude,
    Longitude                 AS longitude
FROM table_1;

num_affected_rows,num_inserted_rows


In [0]:
%sql
CREATE OR REPLACE TABLE dim_customers AS
SELECT DISTINCT
    INT(Order_Customer_Id)    AS customer_id,
    Customer_Fname            AS customer_first_name,
    Customer_Lname            AS customer_last_name,
    Customer_Segment          AS customer_segment,
    
    -- Foreign key linking customer to dim_locations
    CONCAT_WS('-', Market, Order_Region, Order_Country, Customer_State, Customer_City) AS location_id
FROM table_1;

num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT 'fact_orders' AS table_name, COUNT(*) AS row_count FROM fact_orders
UNION ALL
SELECT 'fact_shipments', COUNT(*) FROM fact_shipments
UNION ALL
SELECT 'dim_products', COUNT(*) FROM dim_products
UNION ALL
SELECT 'dim_locations', COUNT(*) FROM dim_locations
UNION ALL
SELECT 'dim_customers', COUNT(*) FROM dim_customers;

table_name,row_count
fact_orders,180519
fact_shipments,180519
dim_products,118
dim_locations,54882
dim_customers,61813


In [0]:
%sql
SELECT 
    l.market,
    p.category_name,
    COUNT(DISTINCT f.order_id)         AS total_orders,
    SUM(f.order_item_quantity)        AS total_units_sold,
    ROUND(SUM(f.order_item_total), 2) AS total_sales_usd
FROM fact_orders f
JOIN dim_products p 
    ON f.product_id = p.product_id
JOIN dim_locations l 
    ON f.location_id = l.location_id
GROUP BY 
    l.market,
    p.category_name
ORDER BY 
    total_sales_usd DESC
LIMIT 10;

market,category_name,total_orders,total_units_sold,total_sales_usd
Europe,Fishing,4244,1686474,6.080715119E8
LATAM,Fishing,4455,1302492,4.7006538511E8
Europe,Cleats,5659,7003269,3.8068393086E8
Europe,Camping & Hiking,3379,1298317,3.5129314587E8
Europe,Cardio Equipment,3185,3767864,3.3377863537E8
Pacific Asia,Fishing,3200,848467,3.0447697043E8
LATAM,Cleats,6057,5557065,2.9899624601E8
Europe,Water Sports,3831,1588505,2.8748681064E8
LATAM,Camping & Hiking,3775,1040201,2.8097889371E8
Europe,Women's Apparel,5034,6210053,2.7872934875E8


In [0]:
%sql
WITH route_metrics AS (
    SELECT 
        l.market,
        l.order_region,
        s.shipping_mode,
        s.days_for_shipping_real AS actual_lead_time,
        s.days_for_shipment_scheduled AS scheduled_lead_time,
        
        -- On-time flag (1 if delivered on or before schedule, else 0)
        CASE WHEN s.days_for_shipping_real <= s.days_for_shipment_scheduled THEN 1 ELSE 0 END AS is_ontime,
        
        -- Late/Backlog flag
        CASE WHEN s.late_delivery_risk = 1 OR s.shipment_status = 'UNFULFILLED_OR_PENDING' THEN 1 ELSE 0 END AS is_backlog_risk
    FROM fact_shipments s
    JOIN fact_orders o ON s.order_item_id = o.order_item_id
    JOIN dim_locations l ON o.location_id = l.location_id
),

route_window_stats AS (
    SELECT 
        market,
        order_region,
        shipping_mode,
        actual_lead_time,
        scheduled_lead_time,
        is_ontime,
        is_backlog_risk,
        
        -- Window Functions across Route + Mode
        COUNT(*) OVER(PARTITION BY market, order_region, shipping_mode) AS total_route_shipments,
        AVG(actual_lead_time) OVER(PARTITION BY market, order_region, shipping_mode) AS avg_lead_time_days,
        
        -- Variance & Standard Deviation (Measures predictability)
        VAR_SAMP(actual_lead_time) OVER(PARTITION BY market, order_region, shipping_mode) AS lead_time_variance,
        STDDEV_SAMP(actual_lead_time) OVER(PARTITION BY market, order_region, shipping_mode) AS lead_time_stddev
    FROM route_metrics
)

SELECT 
    market,
    order_region,
    shipping_mode,
    total_route_shipments,
    
    -- On-time delivery rate percentage
    ROUND(SUM(is_ontime) * 100.0 / total_route_shipments, 2) AS ontime_delivery_rate_pct,
    
    -- Backlog / Stockout risk frequency percentage
    ROUND(SUM(is_backlog_risk) * 100.0 / total_route_shipments, 2) AS backlog_risk_rate_pct,
    
    -- Lead time metrics
    ROUND(AVG(avg_lead_time_days), 2) AS avg_lead_time_days,
    ROUND(AVG(lead_time_variance), 2) AS lead_time_variance,
    ROUND(AVG(lead_time_stddev), 2)   AS lead_time_stddev
FROM route_window_stats
GROUP BY 
    market,
    order_region,
    shipping_mode,
    total_route_shipments
HAVING total_route_shipments >= 50 -- Filter out low-volume noise
ORDER BY lead_time_variance DESC;

market,order_region,shipping_mode,total_route_shipments,ontime_delivery_rate_pct,backlog_risk_rate_pct,avg_lead_time_days,lead_time_variance,lead_time_stddev
Africa,Southern Africa,Standard Class,31621,62.44,35.81,3.99,2.23,1.49
USCA,US Center,Second Class,259519,22.88,75.49,3.93,2.21,1.49
LATAM,South America,Second Class,571377,27.07,70.14,3.74,2.19,1.48
Europe,Eastern Europe,Standard Class,81580,58.47,39.96,4.04,2.16,1.47
USCA,East of USA,Second Class,354067,17.67,78.59,4.09,2.11,1.45
USCA,South of USA,Standard Class,421266,56.95,40.68,4.04,2.11,1.45
Europe,Northern Europe,Standard Class,1719238,58.57,39.66,4.05,2.11,1.45
Pacific Asia,Southeast Asia,Standard Class,846127,58.97,39.29,4.01,2.1,1.45
Pacific Asia,Southeast Asia,Second Class,260394,23.60,73.08,3.88,2.1,1.45
LATAM,Caribbean,Second Class,234786,17.73,77.78,4.12,2.09,1.45


In [0]:
%sql
WITH route_variance_stats AS (
    SELECT 
        l.market,
        l.order_region,
        s.shipping_mode,
        COUNT(*) AS total_shipments,
        ROUND(AVG(s.days_for_shipping_real), 2) AS avg_lead_time_days,
        ROUND(VAR_SAMP(s.days_for_shipping_real), 2) AS lead_time_variance,
        ROUND(STDDEV_SAMP(s.days_for_shipping_real), 2) AS lead_time_stddev,
        ROUND(SUM(CASE WHEN s.days_for_shipping_real <= s.days_for_shipment_scheduled THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS ontime_delivery_rate_pct
    FROM fact_shipments s
    JOIN fact_orders o ON s.order_item_id = o.order_item_id
    JOIN dim_locations l ON o.location_id = l.location_id
    GROUP BY 
        l.market,
        l.order_region,
        s.shipping_mode
    HAVING COUNT(*) >= 50 -- Filter out low-volume routes for statistical significance
),

ranked_routes AS (
    SELECT 
        market,
        order_region,
        shipping_mode,
        total_shipments,
        avg_lead_time_days,
        lead_time_variance,
        lead_time_stddev,
        ontime_delivery_rate_pct,
        -- Rank routes by variance in descending order, resetting rank for each market
        RANK() OVER (
            PARTITION BY market 
            ORDER BY lead_time_variance DESC
        ) AS volatility_rank
    FROM route_variance_stats
)

SELECT 
    market,
    volatility_rank,
    order_region,
    shipping_mode,
    total_shipments,
    lead_time_variance,
    lead_time_stddev,
    avg_lead_time_days,
    ontime_delivery_rate_pct
FROM ranked_routes
WHERE volatility_rank <= 5
ORDER BY 
    market, 
    volatility_rank;

market,volatility_rank,order_region,shipping_mode,total_shipments,lead_time_variance,lead_time_stddev,avg_lead_time_days,ontime_delivery_rate_pct
Africa,1,Southern Africa,Standard Class,31621,2.23,1.49,3.99,62.44
Africa,2,West Africa,Second Class,39877,2.06,1.44,4.03,20.59
Africa,3,Central Africa,Standard Class,35773,2.03,1.42,4.19,54.02
Africa,4,Central Africa,Second Class,11037,2.01,1.42,4.15,18.80
Africa,5,North Africa,Standard Class,79807,1.92,1.38,4.12,58.23
Europe,1,Eastern Europe,Standard Class,81580,2.16,1.47,4.04,58.47
Europe,2,Northern Europe,Standard Class,1719238,2.11,1.45,4.05,58.57
Europe,3,Southern Europe,Second Class,343934,2.09,1.45,3.89,21.59
Europe,3,Western Europe,Standard Class,7529702,2.09,1.44,3.92,60.99
Europe,5,Eastern Europe,Second Class,25022,2.06,1.43,3.94,23.62
